# YOLO26 + EdgeViT

本 Notebook 在 Google Colab 中完成以下工作链：

1. 拉取 Ultralytics YOLO26 和官方 EdgeViT 源码；
2. 检查 YOLO26 基线和 GPU；
3. 兼容新版 timm 的导入接口；
4. 创建 EdgeViT 多尺度骨干适配器；
5. 将 EdgeViT 注册到 Ultralytics；
6. 创建 YOLO26 + EdgeViT YAML；
7. 重启运行时后验证模型构建、前向、反向和 1 epoch 冒烟训练。

第一版使用 EdgeViT_XXS，输出 P3/P4/P5 三层特征。官方 EdgeViT 源码默认要求固定输入尺寸，因此适配器内部统一使用 640×640，再把特征图缩放回 YOLO 当前输入对应的尺寸。


In [1]:
# Colab 新运行时初始化：拉取 YOLO26 和官方 EdgeViT
from pathlib import Path
import subprocess

CONTENT = Path("/content")
ULTRALYTICS = CONTENT / "ultralytics"
EDGEVIT = CONTENT / "edgevit"

def run(cmd, cwd=None):
    print("$", cmd)
    subprocess.run(cmd, cwd=cwd, shell=True, check=True)

if not (ULTRALYTICS / ".git").exists():
    run("git clone --depth 1 https://github.com/ultralytics/ultralytics.git /content/ultralytics")
else:
    print("已有 /content/ultralytics，跳过 clone")

if not (EDGEVIT / ".git").exists():
    run("git clone --depth 1 https://github.com/saic-fi/edgevit.git /content/edgevit")
else:
    print("已有 /content/edgevit，跳过 clone")

run("git fetch --depth 1 origin main", cwd=ULTRALYTICS)
run("git checkout --detach origin/main", cwd=ULTRALYTICS)

assert (ULTRALYTICS / "ultralytics" / "cfg" / "models" / "26" / "yolo26.yaml").exists()
assert (EDGEVIT / "src" / "edgevit.py").exists()

run("python -m pip install -q -e /content/ultralytics --no-deps")

print("YOLO26 commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=ULTRALYTICS, text=True
).strip())
print("EdgeViT commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=EDGEVIT, text=True
).strip())


已有 /content/ultralytics，跳过 clone
已有 /content/edgevit，跳过 clone
$ git fetch --depth 1 origin main
$ git checkout --detach origin/main
$ python -m pip install -q -e /content/ultralytics --no-deps
YOLO26 commit: 23b4a0d9bd9cf1d48a203dfdd0c0112c2ff6f51e
EdgeViT commit: df3d070addc3dc761173429426eab5edeb0d6343


In [2]:
# 建立 YOLO26 基线，确认 Colab、PyTorch、CUDA 和模型加载正常
%cd /content/ultralytics

import sys
import numpy as np
import torch
import ultralytics
from ultralytics import YOLO

repo = "/content/ultralytics"
if repo not in sys.path:
    sys.path.insert(0, repo)

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Ultralytics:", ultralytics.__file__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

baseline = YOLO("yolo26n.pt")
print("YOLO26 baseline loaded")


/content/ultralytics
Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
NumPy: 2.4.6
Ultralytics: /content/ultralytics/ultralytics/__init__.py
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
YOLO26 baseline loaded


In [3]:
# 安装 timm，并记录当前版本。
# EdgeViT 不依赖 timm 中的 EdgeViT 注册名，但官方源码仍使用 timm 的基础层。
%pip install -q timm

import timm

print("timm:", timm.__version__)
print("timm EdgeViT entries:", timm.list_models("*edgevit*"))


timm: 1.0.31
timm EdgeViT entries: ['edgevit_s', 'edgevit_xs', 'edgevit_xxs']


In [4]:
# 兼容官方 EdgeViT 源码与新版 timm 的导入路径
from pathlib import Path
import shutil

edgevit_source = Path("/content/edgevit/src/edgevit.py")
source_text = edgevit_source.read_text(encoding="utf-8")

old_imports = '''from timm.models.vision_transformer import _cfg
from timm.models.registry import register_model
from timm.models.layers import trunc_normal_, DropPath, to_2tuple
'''

new_imports = '''from timm.models.vision_transformer import _cfg

try:
    from timm.models.registry import register_model
except ImportError:
    from timm.models._registry import register_model

try:
    from timm.models.layers import trunc_normal_, DropPath, to_2tuple
except ImportError:
    from timm.layers import trunc_normal_, DropPath, to_2tuple
'''

backup = Path(str(edgevit_source) + ".orig")
if not backup.exists():
    shutil.copy2(edgevit_source, backup)

if old_imports in source_text:
    source_text = source_text.replace(old_imports, new_imports, 1)
elif "from timm.models._registry import register_model" in source_text:
    print("EdgeViT 导入兼容补丁已经存在")
else:
    raise RuntimeError("没有找到预期的 EdgeViT timm 导入块，停止以避免误改源码")

edgevit_source.write_text(source_text, encoding="utf-8")
print("patched:", edgevit_source)


EdgeViT 导入兼容补丁已经存在
patched: /content/edgevit/src/edgevit.py


In [5]:
# 创建 EdgeViT -> YOLO26 的多尺度骨干适配器
from pathlib import Path

root = Path("/content/ultralytics")
extra = root / "ultralytics" / "nn" / "modules" / "extramodules"
extra.mkdir(parents=True, exist_ok=True)
(extra / "__init__.py").touch()

adapter = r'''# Ultralytics AGPL-3.0 License - https://ultralytics.com/license
# EdgeViT adapter for YOLO26.
from __future__ import annotations

import sys
from functools import partial

import torch
from torch import nn
import torch.nn.functional as F

EDGEVIT_SRC = "/content/edgevit/src"
if EDGEVIT_SRC not in sys.path:
    sys.path.insert(0, EDGEVIT_SRC)

from edgevit import EdgeVit


_VARIANTS = {
    "xxs": {
        "depth": [1, 1, 3, 2],
        "embed_dim": [36, 72, 144, 288],
        "head_dim": 36,
        "sr_ratios": [4, 2, 2, 1],
    },
    "xs": {
        "depth": [1, 1, 3, 1],
        "embed_dim": [48, 96, 240, 384],
        "head_dim": 48,
        "sr_ratios": [4, 2, 2, 1],
    },
    "s": {
        "depth": [1, 2, 5, 3],
        "embed_dim": [48, 96, 240, 384],
        "head_dim": 48,
        "sr_ratios": [4, 2, 2, 1],
    },
}


class _EdgeViTBackbone(nn.Module):
    # 输出 EdgeViT 的 stride 8/16/32 特征 [P3, P4, P5]。

    def __init__(self, variant="xxs", img_size=640):
        super().__init__()

        cfg = _VARIANTS[variant]
        self.img_size = img_size
        self.channel = list(cfg["embed_dim"][1:])

        self.model = EdgeVit(
            img_size=img_size,
            num_classes=0,
            mlp_ratio=[4] * 4,
            qkv_bias=True,
            norm_layer=partial(nn.LayerNorm, eps=1e-6),
            **cfg,
        )

    def forward(self, x: torch.Tensor):
        original_h, original_w = x.shape[-2:]

        # 官方 PatchEmbed 会检查输入尺寸。YOLO 构建 stride 时可能传入较小张量，
        # 所以统一到 640×640，完成骨干计算后再缩放回当前输入的 P3/P4/P5 尺寸。
        if (original_h, original_w) != (self.img_size, self.img_size):
            x = F.interpolate(
                x,
                size=(self.img_size, self.img_size),
                mode="bilinear",
                align_corners=False,
            )

        # Stage 1: stride 4
        x = self.model.patch_embed1(x)
        x = self.model.pos_drop(x)
        for block in self.model.blocks1:
            x = block(x)

        # Stage 2: stride 8，作为 P3
        x = self.model.patch_embed2(x)
        for block in self.model.blocks2:
            x = block(x)
        p3 = x

        # Stage 3: stride 16，作为 P4
        x = self.model.patch_embed3(x)
        for block in self.model.blocks3:
            x = block(x)
        p4 = x

        # Stage 4: stride 32，作为 P5
        x = self.model.patch_embed4(x)
        for block in self.model.blocks4:
            x = block(x)
        p5 = self.model.norm(x)

        # 让输出尺寸始终与 YOLO 当前输入对应。
        p3 = F.interpolate(
            p3,
            size=(max(original_h // 8, 1), max(original_w // 8, 1)),
            mode="nearest",
        )
        p4 = F.interpolate(
            p4,
            size=(max(original_h // 16, 1), max(original_w // 16, 1)),
            mode="nearest",
        )
        p5 = F.interpolate(
            p5,
            size=(max(original_h // 32, 1), max(original_w // 32, 1)),
            mode="nearest",
        )

        return [p3, p4, p5]


class EdgeViT_XXS(_EdgeViTBackbone):
    def __init__(self):
        super().__init__("xxs")


class EdgeViT_XS(_EdgeViTBackbone):
    def __init__(self):
        super().__init__("xs")


class EdgeViT_S(_EdgeViTBackbone):
    def __init__(self):
        super().__init__("s")
'''

adapter_path = extra / "edgevit.py"
adapter_path.write_text(adapter, encoding="utf-8")
print("adapter:", adapter_path)


adapter: /content/ultralytics/ultralytics/nn/modules/extramodules/edgevit.py


In [6]:
# 独立验证 EdgeViT 适配器的三个输出尺度
import importlib.util
import torch

adapter_path = "/content/ultralytics/ultralytics/nn/modules/extramodules/edgevit.py"
spec = importlib.util.spec_from_file_location("edgevit_adapter", adapter_path)
edgevit_adapter = importlib.util.module_from_spec(spec)
spec.loader.exec_module(edgevit_adapter)

assert issubclass(edgevit_adapter.EdgeViT_XXS, torch.nn.Module)

device = "cuda" if torch.cuda.is_available() else "cpu"
backbone = edgevit_adapter.EdgeViT_XXS().to(device).eval()
x = torch.randn(1, 3, 640, 640, device=device)

with torch.inference_mode():
    outputs = backbone(x)

print("channels:", backbone.channel)
for index, output in enumerate(outputs):
    print(index, tuple(output.shape))

assert backbone.channel == [72, 144, 288]
assert [tuple(output.shape[-2:]) for output in outputs] == [
    (80, 80),
    (40, 40),
    (20, 20),
]
print("EdgeViT adapter test passed")


channels: [72, 144, 288]
0 (1, 72, 80, 80)
1 (1, 144, 40, 40)
2 (1, 288, 20, 20)
EdgeViT adapter test passed


## 修改 Ultralytics 模块注册和 parse_model

下面的单元会：

- 在 ultralytics.nn.modules 导入 EdgeViT 类；
- 在 tasks.py 中将 EdgeViT 标记为特殊骨干；
- 让解析器读取实例化后的 m_.channel；
- 为原文件创建 .bak 备份；
- 创建 EdgeViT YAML。

完成后必须重启 Colab 运行时，清除已经缓存的 Python 模块。


In [7]:
# 注册 EdgeViT 模块，并对 parse_model 做最小修改
from pathlib import Path
import shutil

root = Path("/content/ultralytics")
modules_init = root / "ultralytics" / "nn" / "modules" / "__init__.py"
tasks_path = root / "ultralytics" / "nn" / "tasks.py"

registration = '''
from .extramodules.edgevit import (
    EdgeViT_XXS,
    EdgeViT_XS,
    EdgeViT_S,
)
'''

modules_text = modules_init.read_text(encoding="utf-8")
if "from .extramodules.edgevit import" not in modules_text:
    modules_backup = Path(str(modules_init) + ".bak")
    if not modules_backup.exists():
        shutil.copy2(modules_init, modules_backup)
    modules_init.write_text(modules_text + registration, encoding="utf-8")
    print("已注册 EdgeViT 到 modules/__init__.py")
else:
    print("modules/__init__.py 已经注册 EdgeViT")

tasks_text = tasks_path.read_text(encoding="utf-8")

if "from ultralytics.nn.modules.extramodules.edgevit import" not in tasks_text:
    import_marker = "from ultralytics.nn.autobackend import check_class_names"
    imports = '''from ultralytics.nn.modules.extramodules.edgevit import (
    EdgeViT_XXS,
    EdgeViT_XS,
    EdgeViT_S,
)
'''
    if import_marker not in tasks_text:
        raise RuntimeError("tasks.py 中没有找到预期导入标记")
    tasks_text = tasks_text.replace(import_marker, imports + "\n" + import_marker, 1)

if "edgevit_modules = frozenset" not in tasks_text:
    loop_marker = '    for i, (f, n, m, args) in enumerate(d["backbone"] + d["head"]):  # from, number, module, args'
    collection = '''    edgevit_modules = frozenset({
        EdgeViT_XXS,
        EdgeViT_XS,
        EdgeViT_S,
    })
'''
    if loop_marker not in tasks_text:
        raise RuntimeError("tasks.py 中没有找到 parse_model 循环")
    tasks_text = tasks_text.replace(loop_marker, collection + loop_marker, 1)

    n_marker = "        n = n_ = max(round(n * depth), 1) if n > 1 else n  # depth gain"
    if n_marker not in tasks_text:
        raise RuntimeError("tasks.py 中没有找到 depth 计算行")
    tasks_text = tasks_text.replace(n_marker, n_marker + "\n        m_ = None", 1)

    branch_marker = "        if m in base_modules:"
    branch = '''        if m in edgevit_modules:
            if n != 1:
                raise ValueError("EdgeViT backbone only supports repeats=1")
            m_ = m(*args)
            c2 = m_.channel
        elif m in base_modules:
'''
    if branch_marker not in tasks_text:
        raise RuntimeError("tasks.py 中没有找到 base_modules 分支")
    tasks_text = tasks_text.replace(branch_marker, branch, 1)

    instantiate = "        m_ = torch.nn.Sequential(*(m(*args) for _ in range(n))) if n > 1 else m(*args)  # module"
    replacement = "        if m_ is None:\n            m_ = torch.nn.Sequential(*(m(*args) for _ in range(n))) if n > 1 else m(*args)  # module"
    if instantiate not in tasks_text:
        raise RuntimeError("tasks.py 中没有找到模块实例化行")
    tasks_text = tasks_text.replace(instantiate, replacement, 1)

    tasks_backup = Path(str(tasks_path) + ".bak")
    if not tasks_backup.exists():
        shutil.copy2(tasks_path, tasks_backup)

    tasks_path.write_text(tasks_text, encoding="utf-8")
    print("已修改 tasks.py")
else:
    print("tasks.py 已经包含 EdgeViT parse_model 分支")

print("模块注册完成。现在请执行：运行时 -> 重新启动运行时")


modules/__init__.py 已经注册 EdgeViT
tasks.py 已经包含 EdgeViT parse_model 分支
模块注册完成。现在请执行：运行时 -> 重新启动运行时


In [8]:
# 创建第一版 YOLO26 + EdgeViT 配置
from pathlib import Path

cfg_path = Path(
    "/content/ultralytics/ultralytics/cfg/models/26/yolo26_edgevit.yaml"
)

cfg_path.write_text(
'''# YOLO26 with EdgeViT-XXS backbone.
# EdgeViT-XXS returns [P3, P4, P5] with channels [72, 144, 288].

nc: 80
end2end: True
reg_max: 1

backbone:
  - [-1, 1, EdgeViT_XXS, []]

head:
  - [0, 1, Index, [72, 0]]
  - [0, 1, Index, [144, 1]]
  - [0, 1, Index, [288, 2]]
  - [[1, 2, 3], 1, Detect, [nc]]
''',
    encoding="utf-8",
)

print("YAML:", cfg_path)


YAML: /content/ultralytics/ultralytics/cfg/models/26/yolo26_edgevit.yaml


## 重启运行时

执行上一个单元后，请在 Colab 菜单中选择：

运行时 → 重新启动运行时

重启后，从初始化单元重新执行到这里。这样可以清除旧的 ultralytics 和 tasks.py 导入缓存。


In [9]:
# 重启运行时后，构建 YOLO26 + EdgeViT 模型
%cd /content/ultralytics

from ultralytics import YOLO

model = YOLO(
    "/content/ultralytics/ultralytics/cfg/models/26/yolo26_edgevit.yaml"
)

model.info(verbose=True)
print("model build passed")


/content/ultralytics
YOLO26_edgevit summary: 241 layers, 5,072,544 parameters, 5,072,544 gradients
model build passed


In [10]:
# 前向传播验证
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.model = model.model.to(device).eval()

dummy = torch.randn(1, 3, 640, 640, device=device)

with torch.inference_mode():
    raw_outputs = model.model(dummy)

print("forward passed")


forward passed


In [ ]:
# 反向传播测试
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

net = model.model.to(device)
net.train()
net.zero_grad(set_to_none=True)

x = torch.rand(1, 3, 640, 640, device=device)
outputs = net(x)

def collect_tensors(value):
    if isinstance(value, torch.Tensor):
        return [value]

    if isinstance(value, dict):
        tensors = []
        for item in value.values():
            tensors.extend(collect_tensors(item))
        return tensors

    if isinstance(value, (list, tuple)):
        tensors = []
        for item in value:
            tensors.extend(collect_tensors(item))
        return tensors

    return []

tensors = [
    tensor for tensor in collect_tensors(outputs)
    if tensor.requires_grad
]

print("output type:", type(outputs))
if isinstance(outputs, dict):
    print("output keys:", list(outputs.keys()))

print("output tensors:", len(tensors))
print("requires_grad:", [tensor.requires_grad for tensor in tensors])

assert tensors, "没有找到可反向传播的 Tensor"

loss = torch.stack([
    tensor.float().mean()
    for tensor in tensors
]).sum()

print("loss:", loss.item())
loss.backward()

print("backward passed")

output type: <class 'dict'>
output keys: ['one2many', 'one2one']
output tensors: 7
requires_grad: [True, True, True, True, True, True, True]
loss: -18.363500595092773
backward passed


In [ ]:
# 1 epoch 冒烟训练
# 该步骤只用于确认数据加载、损失计算、反向传播和权重保存链路。
model.train(
    data="coco8.yaml",
    epochs=1,
    imgsz=640,
    batch=2,
    workers=2,
    device=0,
    project="/content/runs",
    name="yolo26_edgevit_smoke",
)


Ultralytics 8.4.207 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/ultralytics/runs/detect/train/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26_edgevit_smoke, nbs=64, nms=N

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0, 16, 17, 20, 25, 58])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7992f0f93af0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

In [ ]:
# 检查训练结果和工作区改动
from pathlib import Path
import subprocess

best = Path("/content/runs/yolo26_edgevit_smoke/weights/best.pt")
print("best checkpoint:", best, "exists:", best.exists())

print("\nUltralytics working tree:")
subprocess.run(["git", "status", "--short"], cwd="/content/ultralytics", check=False)

print("\nDiff summary:")
subprocess.run(["git", "diff", "--stat"], cwd="/content/ultralytics", check=False)


best checkpoint: /content/runs/yolo26_edgevit_smoke/weights/best.pt exists: True

Ultralytics working tree:

Diff summary:


CompletedProcess(args=['git', 'diff', '--stat'], returncode=0)